# Reconstructed token-level MG scores

Compare saved LittleLamb Logistic Regression and Random Forest classifiers at a 64-token window on selected SemEval, Name That LLM, and BBC News passages. Window scores are projected onto token positions by averaging overlapping window scores that cover each token.

SemEval examples include the annotated HG/MG boundary. Name That LLM examples are MG-only: source-LLM identity is shown, but no gold token boundary is implied. BBC examples are HG-only and show the classifiers' MG probabilities, without a gold boundary. The notebook reads cached features, saved classifier artifacts, source passages, and Stage-1 token surfaces; it does not fit classifiers, run inference, or retokenize text.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from notebooks.utils.reconstructed_token_mg_score_visualization_utils import (
    WINDOW_CONFIGURATION,
    build_excerpt_rows,
    load_text_records,
    load_visualization_models,
    plot_passage_scores,
    plot_passage_score_dispersion,
    plot_window_boundary_profile,
    reconstruct_token_scores,
    reconstruct_token_score_dispersion,
    detect_window_run_boundaries,
    save_visualization_outputs,
    score_prepared_partition,
    select_visualization_examples,
    semeval_boundary_errors_from_scored_windows,
)

DATA_ROOT = Path(os.environ.get('TEXT_DETECTION_DATA', 'D:/Projects/master_diploma/text-detection-data'))
SCORER_LLM = "littlelamb"
MODEL_RUN_ID = os.environ.get(
    'CLASSIFIER_RUN_ID',
    'feature-vector-classification-2026-09-25-logreg-random-forest-max600-min100',
)
PLOT_RUN_ID = os.environ.get('PLOT_RUN_ID', 'reconstructed-token-mg-scores-2026-10-29')
FEATURE_SET = 'simple_similarity'
TRAINING_REGIME = 'Kaggle+MAGE+Outfox'
WINDOW_CONFIGURATION = 64
WINDOW_BOUNDARY_THRESHOLD = 0.5
WINDOW_BOUNDARY_RUN_LENGTH = 3
SELECTION_SEED = 42
EXCERPT_WIDTH = 80
MANUAL_TEXT_IDS = {'semeval': [], 'name_that_llm_train': [], 'bbc': []}
MANUAL_EXCERPT_STARTS = {}  # Optional: {(dataset, text_id): first_token_position}
DATASETS = {
    'semeval': 'semeval2024_task8_subtask_c_train',
    'name_that_llm_train': 'name_that_llm_train',
    'bbc': 'bbc_news_content',
}
OUTPUT_ROOT = DATA_ROOT / 'plots' / PLOT_RUN_ID / 'reconstructed_token_mg_scores'
print({
    'data_root': DATA_ROOT, 'scorer_llm': SCORER_LLM, 'model_run_id': MODEL_RUN_ID,
    'models': ('logistic_regression', 'random_forest'), 'window_configuration': WINDOW_CONFIGURATION,
    'feature_set': FEATURE_SET, 'training_regime': TRAINING_REGIME, 'output_root': OUTPUT_ROOT,
})

## Load saved classifiers and cached windows

The configured run ID must identify one saved LR and RF artifact for LittleLamb at the 64-token window. Prepared partitions are read-only inputs; missing caches or artifacts are reported rather than regenerated.

In [ ]:
models = load_visualization_models(
    DATA_ROOT, run_id=MODEL_RUN_ID, scorer_llm=SCORER_LLM, training_regime=TRAINING_REGIME,
)
source_records = {}
scored_window_frames = []
for dataset_kind, dataset_name in DATASETS.items():
    records = load_text_records(DATA_ROOT, dataset=dataset_name, scorer_llm=SCORER_LLM)
    records['dataset'] = dataset_kind
    source_records[dataset_kind] = records
    for model_family, model in models.items():
        split = 'HG' if dataset_kind == 'bbc' else 'MG'
        scored = score_prepared_partition(
            DATA_ROOT, dataset=dataset_name, split=split, model=model, scorer_llm=SCORER_LLM,
        )
        scored['dataset'] = dataset_kind
        scored_window_frames.append(scored)
scored_windows = pd.concat(scored_window_frames, ignore_index=True)

record_lookup = {
    dataset_kind: records.set_index('text_id', drop=False)
    for dataset_kind, records in source_records.items()
}
token_score_frames = []
for (dataset_kind, text_id), windows in scored_windows.groupby(['dataset', 'text_id'], sort=True):
    records = record_lookup[dataset_kind]
    if text_id not in records.index:
        raise KeyError(f'raw text or Stage-1 tokens missing for {dataset_kind}/{text_id}')
    record = records.loc[text_id]
    projected = reconstruct_token_scores(
        windows, text_id=text_id, token_count=int(record['token_count']),
    )
    projected['dataset'] = dataset_kind
    token_score_frames.append(projected)
token_scores = pd.concat(token_score_frames, ignore_index=True)

semeval_boundary_errors = semeval_boundary_errors_from_scored_windows(
    scored_windows.loc[scored_windows['dataset'].eq('semeval')],
    source_records['semeval'],
    threshold=0.5, smoothing_window=5, sustained_length=3,
)
semeval_boundary_exclusions = semeval_boundary_errors.attrs['exclusions']
if not semeval_boundary_exclusions.empty:
    print(f'Excluded {semeval_boundary_exclusions.text_id.nunique()} unlabelable SemEval texts from boundary ranking')
    display(semeval_boundary_exclusions.groupby('exclusion_reason').text_id.nunique().rename('text_count').reset_index())
display(pd.DataFrame({
    dataset: {'texts': records['text_id'].nunique(), 'tokenized texts': len(records)}
    for dataset, records in source_records.items()
}).T)

## Select examples and inspect score/text alignment

Automatic SemEval examples span the best, typical, and worst boundary localization derived from this run's saved-model scores using the standard smoothed threshold/sustained-run rule. Name That LLM examples span low, middle, and high mean scores and include the strongest LR/RF token-score disagreement. BBC examples span the lowest, median-nearest, and highest mean reconstructed MG probabilities across the HG texts. Add IDs in `MANUAL_TEXT_IDS` to include curated examples.

In [ ]:
selection = select_visualization_examples(
    token_scores,
    semeval_boundary_errors=semeval_boundary_errors,
    manual_ids=MANUAL_TEXT_IDS,
    seed=SELECTION_SEED,
)
selection = selection.merge(
    pd.concat(source_records.values(), ignore_index=True)[['dataset', 'text_id', 'source_llm'] if 'source_llm' in pd.concat(source_records.values(), ignore_index=True).columns else ['dataset', 'text_id']].drop_duplicates(['dataset', 'text_id']),
    on=['dataset', 'text_id'], how='left', validate='many_to_one',
)
figures = {}
for example in selection.to_dict(orient='records'):
    dataset_kind = example['dataset']
    text_id = str(example['text_id'])
    record = record_lookup[dataset_kind].loc[text_id]
    current_scores = token_scores.loc[
        token_scores['dataset'].eq(dataset_kind) & token_scores['text_id'].astype(str).eq(text_id)
    ].copy()
    gold_boundary = None
    if dataset_kind == 'semeval':
        boundary_rows = semeval_boundary_errors.loc[semeval_boundary_errors['text_id'].astype(str).eq(text_id)]
        if boundary_rows.empty:
            raise KeyError(f'gold/predicted SemEval boundary result unavailable for {text_id}')
        gold_boundary = int(boundary_rows['gold_token_boundary'].iloc[0])
        example['gold_token_boundary'] = gold_boundary
    excerpt_start = MANUAL_EXCERPT_STARTS.get((dataset_kind, text_id))
    excerpt = build_excerpt_rows(
        current_scores, record['tokens'], dataset=dataset_kind, excerpt_width=EXCERPT_WIDTH,
        gold_boundary=gold_boundary, excerpt_start=excerpt_start,
    )
    figure = plot_passage_scores(
        example, current_scores, excerpt,
        title=f'LittleLamb · 64-token windows · {example["selection_group"]}',
    )
    figure_key = f'{dataset_kind}__{text_id}__{example["selection_group"]}'
    figures[figure_key] = figure
    display(figure)
    plt.close(figure)
display(selection)

## Token-score dispersion across overlapping windows

For every selected example, this companion plot shows each model's token-level mean MG score as a line and the population standard deviation across windows covering that token as a shaded mean ± SD band. The displayed band is clipped to the valid probability range [0, 1].

In [ ]:
for example in selection.to_dict(orient='records'):
    dataset_kind = example['dataset']
    text_id = str(example['text_id'])
    record = record_lookup[dataset_kind].loc[text_id]
    windows = scored_windows.loc[
        scored_windows['dataset'].eq(dataset_kind)
        & scored_windows['text_id'].astype(str).eq(text_id)
    ].copy()
    dispersion = reconstruct_token_score_dispersion(
        windows, text_id=text_id, token_count=int(record['token_count']),
    )
    figure = plot_passage_score_dispersion(
        dispersion,
        title=f'LittleLamb · 64-token windows · {example["selection_group"]} · mean ± SD',
    )
    figure_key = f'{dataset_kind}__{text_id}__{example["selection_group"]}__dispersion'
    figures[figure_key] = figure
    display(figure)
    plt.close(figure)

## SemEval boundaries from consecutive window scores

A window is HG below the configured threshold and MG at or above it. The first run of three HG windows followed immediately by three MG windows defines a boundary halfway between the centers of the last HG and first MG windows. This is calculated separately for LR and RF and compared with the gold SemEval boundary.

In [ ]:
window_boundary_rows = []
for example in selection.loc[selection['dataset'].eq('semeval')].to_dict(orient='records'):
    text_id = str(example['text_id'])
    windows = scored_windows.loc[
        scored_windows['dataset'].eq('semeval')
        & scored_windows['text_id'].astype(str).eq(text_id)
    ].copy()
    gold_rows = semeval_boundary_errors.loc[
        semeval_boundary_errors['text_id'].astype(str).eq(text_id), 'gold_token_boundary'
    ].dropna()
    gold_boundaries = gold_rows.unique()
    if len(gold_boundaries) != 1:
        raise ValueError(f'Expected one gold SemEval boundary for text {text_id}; found {len(gold_boundaries)}')
    gold_boundary = int(gold_boundaries[0])
    predictions = detect_window_run_boundaries(
        windows, threshold=WINDOW_BOUNDARY_THRESHOLD, run_length=WINDOW_BOUNDARY_RUN_LENGTH,
    )
    predictions['dataset'] = 'semeval'
    predictions['text_id'] = text_id
    predictions['selection_group'] = example['selection_group']
    predictions['gold_token_boundary'] = gold_boundary
    predictions['absolute_token_error'] = (predictions['predicted_token_boundary'] - gold_boundary).abs()
    window_boundary_rows.append(predictions)
    figure = plot_window_boundary_profile(
        windows, predictions, gold_boundary=gold_boundary, threshold=WINDOW_BOUNDARY_THRESHOLD,
        title=f'SemEval {text_id} · {SCORER_LLM} · window-level {WINDOW_BOUNDARY_RUN_LENGTH}+{WINDOW_BOUNDARY_RUN_LENGTH} boundary',
    )
    figure_key = f'semeval__{text_id}__{example["selection_group"]}__window_boundary'
    figures[figure_key] = figure
    display(figure)
    plt.close(figure)
window_boundary_predictions = pd.concat(window_boundary_rows, ignore_index=True)
display(window_boundary_predictions[[
    'text_id', 'selection_group', 'model', 'status', 'predicted_token_boundary',
    'gold_token_boundary', 'absolute_token_error',
]].sort_values(['text_id', 'model']).reset_index(drop=True))

## Custom SemEval visualization

Set one SemEval text ID, the first token position to show, and the excerpt width in tokens. Run this cell after `Load saved classifiers and cached windows` and `Select examples and inspect score/text alignment` to render just that passage.

In [ ]:
CUSTOM_SEMEVAL_TEXT_ID = '20'
CUSTOM_EXCERPT_START_TOKEN = 100
CUSTOM_EXCERPT_WIDTH = 100

custom_text_id = str(CUSTOM_SEMEVAL_TEXT_ID).strip()
if custom_text_id not in record_lookup['semeval'].index:
    raise KeyError(f'SemEval text ID {custom_text_id!r} is unavailable in the loaded data')
custom_record = record_lookup['semeval'].loc[custom_text_id]
custom_scores = token_scores.loc[
    token_scores['dataset'].eq('semeval')
    & token_scores['text_id'].astype(str).eq(custom_text_id)
].copy()
custom_boundary_rows = semeval_boundary_errors.loc[
    semeval_boundary_errors['text_id'].astype(str).eq(custom_text_id)
]
if custom_boundary_rows.empty:
    raise KeyError(f'Gold SemEval boundary unavailable for text ID {custom_text_id!r}')
custom_example = {
    'dataset': 'semeval',
    'text_id': custom_text_id,
    'selection_group': 'custom',
    'gold_token_boundary': int(custom_boundary_rows['gold_token_boundary'].iloc[0]),
}
custom_excerpt = build_excerpt_rows(
    custom_scores, custom_record['tokens'], dataset='semeval',
    excerpt_width=CUSTOM_EXCERPT_WIDTH,
    gold_boundary=int(custom_boundary_rows['gold_token_boundary'].iloc[0]),
    excerpt_start=CUSTOM_EXCERPT_START_TOKEN,
)
custom_figure = plot_passage_scores(
    custom_example, custom_scores, custom_excerpt,
    title=f'LittleLamb · 64-token windows · custom SemEval text {custom_text_id}',
)
display(custom_figure)
plt.close(custom_figure)

In [ ]:
CUSTOM_SEMEVAL_TEXT_ID = '10'
CUSTOM_EXCERPT_START_TOKEN = 60
CUSTOM_EXCERPT_WIDTH = 150

custom_text_id = str(CUSTOM_SEMEVAL_TEXT_ID).strip()
if custom_text_id not in record_lookup['semeval'].index:
    raise KeyError(f'SemEval text ID {custom_text_id!r} is unavailable in the loaded data')
custom_record = record_lookup['semeval'].loc[custom_text_id]
custom_scores = token_scores.loc[
    token_scores['dataset'].eq('semeval')
    & token_scores['text_id'].astype(str).eq(custom_text_id)
].copy()
custom_boundary_rows = semeval_boundary_errors.loc[
    semeval_boundary_errors['text_id'].astype(str).eq(custom_text_id)
]
if custom_boundary_rows.empty:
    raise KeyError(f'Gold SemEval boundary unavailable for text ID {custom_text_id!r}')
custom_example = {
    'dataset': 'semeval',
    'text_id': custom_text_id,
    'selection_group': 'custom',
    'gold_token_boundary': int(custom_boundary_rows['gold_token_boundary'].iloc[0]),
}
custom_excerpt = build_excerpt_rows(
    custom_scores, custom_record['tokens'], dataset='semeval',
    excerpt_width=CUSTOM_EXCERPT_WIDTH,
    gold_boundary=int(custom_boundary_rows['gold_token_boundary'].iloc[0]),
    excerpt_start=CUSTOM_EXCERPT_START_TOKEN,
)
custom_figure = plot_passage_scores(
    custom_example, custom_scores, custom_excerpt,
    title=f'LittleLamb · 64-token windows · custom SemEval text {custom_text_id}',
)
display(custom_figure)
plt.close(custom_figure)

In [ ]:
CUSTOM_SEMEVAL_TEXT_ID = '15'
CUSTOM_EXCERPT_START_TOKEN = 160
CUSTOM_EXCERPT_WIDTH = 50

custom_text_id = str(CUSTOM_SEMEVAL_TEXT_ID).strip()
if custom_text_id not in record_lookup['semeval'].index:
    raise KeyError(f'SemEval text ID {custom_text_id!r} is unavailable in the loaded data')
custom_record = record_lookup['semeval'].loc[custom_text_id]
custom_scores = token_scores.loc[
    token_scores['dataset'].eq('semeval')
    & token_scores['text_id'].astype(str).eq(custom_text_id)
].copy()
custom_boundary_rows = semeval_boundary_errors.loc[
    semeval_boundary_errors['text_id'].astype(str).eq(custom_text_id)
]
if custom_boundary_rows.empty:
    raise KeyError(f'Gold SemEval boundary unavailable for text ID {custom_text_id!r}')
custom_example = {
    'dataset': 'semeval',
    'text_id': custom_text_id,
    'selection_group': 'custom',
    'gold_token_boundary': int(custom_boundary_rows['gold_token_boundary'].iloc[0]),
}
custom_excerpt = build_excerpt_rows(
    custom_scores, custom_record['tokens'], dataset='semeval',
    excerpt_width=CUSTOM_EXCERPT_WIDTH,
    gold_boundary=int(custom_boundary_rows['gold_token_boundary'].iloc[0]),
    excerpt_start=CUSTOM_EXCERPT_START_TOKEN,
)
custom_figure = plot_passage_scores(
    custom_example, custom_scores, custom_excerpt,
    title=f'LittleLamb · 64-token windows · custom SemEval text {custom_text_id}',
)
display(custom_figure)
plt.close(custom_figure)

## Export figures

The exporter refuses to replace an existing run folder. Set a new `PLOT_RUN_ID` to save another set of figures.

In [ ]:
export_paths = save_visualization_outputs(figures, selection, OUTPUT_ROOT)
print(f'Wrote {len(figures)} figures and selection metadata under {OUTPUT_ROOT}')
display(pd.DataFrame({'path': [str(path) for path in export_paths]}))